In [41]:
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

In [42]:
# 1. Load the pre-trained, production-ready model
loaded_model = joblib.load("gradient_boosting_churn_model.pkl")
print("Model loaded successfully!")

Model loaded successfully!


In [43]:
# 2. Load the raw dataset
df = pd.read_csv("data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [44]:
# 3. Standardize targeted lowercase columns to PascalCase using dictionary comprehension
cols_to_capitalize = ['gender', 'tenure', 'churn']
df.rename(columns={c: c.capitalize() for c in cols_to_capitalize}, inplace=True)

In [45]:
# 4. Standardize specific outliers manually for a seamless schema match
df.rename(columns={'customerID': 'CustomerId', 'SeniorCitizen': 'SeniorCitizen'}, inplace=True)

In [46]:
# 5. Handle missing values in TotalCharges (matching the training phase data cleaning)
df['TotalCharges'] = df['TotalCharges'].replace(' ', '0')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

In [47]:
# 6. Feature Engineering (ChargesRatio and IsFamily)
df['ChargesRatio'] = df['TotalCharges'] / (df['MonthlyCharges'] + 0.001)

if 'Partner' in df.columns and 'Dependents' in df.columns:
    df['IsFamily'] = ((df['Partner'] == 'Yes') | (df['Dependents'] == 'Yes')).astype(int)
else:
    df['IsFamily'] = 0

In [48]:
if 'CustomerId' in df.columns:
    df.drop(columns=['CustomerId'], inplace=True)

In [49]:
# 7. Apply One-Hot Encoding identically to the training pipeline
df_encoded = pd.get_dummies(df, drop_first=True)

In [50]:
# 8. Isolate Features (X) by dropping the target column
X = df_encoded.drop(columns=['Churn_Yes'])

In [51]:
# 9. Recreate the exact split to extract aligned real-world customer test cases
_, X_test, _, _ = train_test_split(X, df_encoded['Churn_Yes'], test_size=0.2, random_state=42)

In [52]:
print("Data pipeline successfully aligned with the model schema!")
print(f"Total required features matched: {X_test.shape[1]}")

Data pipeline successfully aligned with the model schema!
Total required features matched: 32


In [53]:
# 1. Select the 5th customer from the cleaned test set
single_customer = X_test.iloc[[5]]

# 2. Align the features strictly with the model's expected order
aligned_customer = single_customer[loaded_model.feature_names_in_]

# 3. Run the inference
prediction = loaded_model.predict(aligned_customer)
probability = loaded_model.predict_proba(aligned_customer)

print("--- Churn Prediction Inference ---")
if prediction[0] == 1:
    print(f"Customer Status: At Risk (Likely to Churn)")
else:
    print(f"Customer Status: Safe (Likely to Stay)")

print(f"Calculated Churn Probability: {round(probability[0][1] * 100, 2)}%")

--- Churn Prediction Inference ---
Customer Status: Safe (Likely to Stay)
Calculated Churn Probability: 15.8%
